# LoRA and QLoRA — Efficient Fine-tuning

Fine-tuning a 7B model updates all 7 billion parameters → needs ~56GB GPU RAM. Not practical.

**LoRA (Low-Rank Adaptation):** instead of updating the full weight matrix W (huge), add two small matrices A and B (tiny) and train only those.

```
Normal: W_new = W + ΔW          ΔW is full size [d_out, d_in]
LoRA:   W_new = W + A @ B       A is [d_out, r], B is [r, d_in]  where r << d
```

If d=4096 and r=16: ΔW has 4096×4096=16M params. A+B has 2×4096×16=131K params. **99% fewer trainable params.**

**QLoRA** = Quantize base model to 4-bit (even less memory) + apply LoRA on top.

**Install:** `pip install peft bitsandbytes transformers`

In [ ]:
# --- LoRA from scratch — understand the math ---
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    """A linear layer with LoRA adapters added on top"""
    def __init__(self, in_features, out_features, rank=16, alpha=32):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.linear.weight.requires_grad = False   # FREEZE the original weights

        # LoRA matrices — these are the only things we train
        self.lora_A = nn.Linear(in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_features, bias=False)

        # scaling factor — alpha/rank controls the magnitude of LoRA update
        self.scale = alpha / rank

        # Initialize: A = random, B = zeros (so LoRA output starts at 0)
        nn.init.kaiming_uniform_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        base_output = self.linear(x)           # original frozen weights
        lora_output = self.lora_B(self.lora_A(x)) * self.scale   # A @ B path
        return base_output + lora_output       # combined output


# Compare parameter counts
layer      = LoRALinear(512, 512, rank=16)
total      = sum(p.numel() for p in layer.parameters())
trainable  = sum(p.numel() for p in layer.parameters() if p.requires_grad)
frozen     = total - trainable

print(f'Total params:     {total:,}')
print(f'Trainable (LoRA): {trainable:,}  ({trainable/total*100:.1f}%)')
print(f'Frozen (base):    {frozen:,}   ({frozen/total*100:.1f}%)')

In [ ]:
# --- Using PEFT library (production approach) ---
# pip install peft

from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load base model
model = AutoModelForCausalLM.from_pretrained('gpt2')

# Define LoRA config
lora_config = LoraConfig(
    r=16,                            # rank — higher = more capacity, more params
    lora_alpha=32,                   # scaling = alpha / r
    target_modules=['c_attn'],       # which layers to apply LoRA to (attention in GPT-2)
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM
)

# Wrap model with LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # shows how few params we're training

In [ ]:
# --- QLoRA — 4-bit quantization + LoRA ---
# pip install bitsandbytes

# This is how you'd load a 7B model on a single 16GB GPU
# (change model_name to 'meta-llama/Llama-2-7b-hf' for a real LLM)

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                         # quantize base model to 4-bit
    bnb_4bit_quant_type='nf4',                 # NormalFloat4 — better than int4 for LLMs
    bnb_4bit_use_double_quant=True,            # quantize the quantization constants too
    bnb_4bit_compute_dtype=torch.bfloat16,     # compute in bfloat16 even though stored in 4-bit
)

# Load base model in 4-bit
# model = AutoModelForCausalLM.from_pretrained(
#     'meta-llama/Llama-2-7b-hf',
#     quantization_config=bnb_config,
#     device_map='auto'   # auto-distribute across available GPUs
# )
#
# Then apply LoRA on top:
# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()   # ~0.1% trainable

print('QLoRA memory savings:')
print('  7B model float32:  ~28 GB')
print('  7B model bfloat16: ~14 GB')
print('  7B model 4-bit:    ~ 4 GB')
print('  7B QLoRA trainable params: ~20-40 MB')